In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import DoubleType, IntegerType, StringType
from pyspark.sql.window import Window

**Using Unity Catalog - created a schema (In unity catalog created credentials, external location and used access connector to connect to source)**

In [0]:
%skip
%sql
show catalogs;
create schema databricks_projects.patientrecord ;
show schemas in databricks_projects ;

**The file is in parquet format; wecan create external table but I tried another option as it is widely used in bronze layer were data can be in semi structured formats or a list of parquet files ; volume is created instead of table** 

Databricks External Volumes and Tables (using Unity Catalog) enable governance over data stored in your own cloud storage (Azure Data Lake Storage) rather than Databricks-managed storage. External volumes manage non-tabular, unstructured files (e.g., CSV, images, JARs), while external tables register structured Delta or Parquet data

In [0]:
%skip
%sql
CREATE EXTERNAL VOLUME databricks_projects.patientrecord.my_volume
LOCATION 'abfss://patient-record@retailsalesgb.dfs.core.windows.net/bronze/';

**Read the required file using the external volume**

In [0]:
df = spark.read.parquet("dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet")
df = (
    df
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_source_file_name", F.col("_metadata.file_name"))
    .withColumn("_file_modification_time", F.col("_metadata.file_modification_time"))
)
display(df.take(10))
SILVER_TABLE = "databricks_projects.silver.patient_records_clean"

patient_id,first_name,last_name,date_of_birth,age,gender,ssn,phone_number,email,address,city,state,zip_code,admission_date,discharge_date,length_of_stay_days,admission_type,department,attending_physician,primary_diagnosis,secondary_diagnosis,medication_1,medication_2,height_cm,weight_kg,bmi,blood_pressure,heart_rate_bpm,temperature_celsius,oxygen_saturation_pct,blood_glucose_mgdl,creatinine_mgdl,hba1c_pct,insurance_provider,insurance_claim_amount_usd,icu_days,readmission_30day_flag,smoker,alcohol_use,discharge_disposition,_ingestion_timestamp,_source_file,_source_file_name,_file_modification_time
93810,Michelle,Mitchell,1961-12-20,58,M,532-53-5552,359-420-6514,michelle.mitchell14@gmail.com,5636 Hill Blvd,Fort Worth,FL,59797,2020-07-03,2020-07-08,5,Urgent,Orthopedics,Dr. Rivera,Type 2 Diabetes,Pneumonia,Amlodipine,Hydrochlorothiazide,151.4,53.0,23.1,167/61,null,37.8,92,73,6.19,5.4,Humana,70584.69,0,1,No,None,Deceased,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
83000,Betty,Moore,1945-01-06,75 yrs,male,697-61-6930,424-341-9348,betty.moore64@gmail.com,2505 Maple Way,Phoenix,OK,null,2020-07-27,2020-07-24,-3,Emergency,Neurology,Dr. Baker,Migraine,Asthma,Hydrochlorothiazide,null,192.4,119.5,32.3,173/91,108,36.6 C,99,345,2.47,9.3,Anthem,21744.49,0,0,0,Moderate,Rehab Facility,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
24662,Andrew,Garcia,1967-08-03,-999,M,873-98-4295,930-519-7537,andrew.garcia86@hotmail.com,7398 Oak Ave,Seattle,CT,67422,2020-09-07,2020-09-09,2,Elective,Cardiology,Dr. Perez,Hyperlipidemia,null,Metformin,Albuterol,186.7,N/A,36.8,106/68,120,38.1,97,286,N/A,12.4,BlueCross BlueShield,6498.92,2,0,No,null,Home,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
24322,SANDRA,THOMAS,1980-03-31,39,M,102-59-5345,665-492-7930,sandra.thomas90aol.com,2537 Cedar Blvd,San Francisco,SC,null,2019-07-29,2019-08-12,14,Transfer,Cardiology,Dr. Allen,Atrial Fibrillation,UTI,Pantoprazole,null,188.8,91.8,25.8,159/60,80,37.9,91,-999,7.27,5.5,BlueCross BlueShield,49863.01,2,0,No,Occasional,Discharged to SNF,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
PT-19508,Susan,Allen,09/23/1995,28,F,***-**-2697,3374702891,susan.allen14@aol.com,9910 Cedar Rd,Austin,NC,30374,2024-04-25,2024-04-29,4,Urgent,Pediatrics,Dr. Hernandez,Anxiety Disorder,null,Gabapentin,Gabapentin,165.7,null,18.5,126/70,119,null,100,340,0.56,11.4,Medicaid,70579.91,0,0,1,null,Discharged to SNF,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
63264,Anthony,Johnson,1961-03-28,58,M,458-49-4728,4282244164,anthony.johnson52@hotmail.com,4574 Elm Rd,Houston,OK,46517,2019-10-28,2019-11-23,26,Emergency,ICU,Dr. Thompson,Stroke,Pneumonia,Levothyroxine,Amlodipine,164.9,124.1,45.6,121/77,null,36.2,95,183,2.0,12.3,Centene,81834.15,2,0,Y,Heavy,Deceased,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
50538,anthony,Nelson,2002-07-20,18,F,318758752,953-373-2389,anthony.nelson37@aol.com,5492 Oak Way,Oklahoma City,CO,null,2021-04-26,2021-05-14,18,Transfer,General Medicine,Dr. Clark,Parkinson's Disease,null,Sertraline,null,190.6,97.3 kg,26.8,160/60,null,38.3,98,234,3.99,8.0,BlueCross BlueShield,4405.06,1,0,Unknown,null,Home,2026-05-06T00:57:16.972341Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z
88047,Steven,Ramirez,1979-09-15,45,F,443-50-9849,282-341-3471,null,1053 Washington Dr,San Diego,SC,30028,202

**Managed Table for storing the cleaned data**

In [0]:
%skip
spark.sql(f"CREATE SCHEMA IF NOT EXISTS databricks_projects.silver")
SILVER_TABLE = "databricks_projects.silver.patient_records_clean"

Managed Table to merge data in gold layer

In [0]:
%skip
spark.sql(f"CREATE SCHEMA IF NOT EXISTS databricks_projects.gold")
GOLD_TABLE = "databricks_projects.gold.patient_records_final"


**Data Cleaning**

Patient ID

Input Format - PT-19508, 00034824

Required Value - 19508, 34824 

Remove duplicates

In [0]:
df = df.withColumn(
    "patient_id_clean",
    F.regexp_replace(F.col("patient_id"), r"^(PT-|0+)", "").cast(IntegerType())
)

window_dedup = (
    Window.partitionBy("patient_id_clean")
          .orderBy(F.col("_file_modification_time").desc())
)
df = (
    df.withColumn("_row_rank", F.row_number().over(window_dedup))
      .filter(F.col("_row_rank") == 1)
      .drop("_row_rank")
)

print(f"After dedup: {df.count():,} rows")


After dedup: 952 rows


**Name**

In [0]:
def clean_name(col_name):
    col = F.trim(F.col(col_name))
    
    return F.when(
        col.isNull() |
        F.upper(col).isin("", "N/A", "NA", "NULL"),
        F.lit(None)
    ).otherwise(
        F.initcap(col)
    )

df = (
    df
    .withColumn("first_name_clean", clean_name("first_name"))
    .withColumn("last_name_clean",  clean_name("last_name"))
    .withColumn("full_name", F.concat_ws(" ", "first_name_clean", "last_name_clean"))
)

**Date**

In [0]:
def parse_date(col_name):
    col = F.trim(F.col(col_name))
    
    return F.coalesce(
        F.when(col.rlike(r"^\d{4}-\d{2}-\d{2}$"),
               F.to_date(col, "yyyy-MM-dd")),
        F.when(col.rlike(r"^\d{2}/\d{2}/\d{4}$"),
               F.to_date(col, "MM/dd/yyyy")),
        F.when(col.rlike(r"^\d{4}/\d{2}/\d{2}$"),
               F.to_date(col, "yyyy/MM/dd"))
    )

df = (
    df
    .withColumn("dob_clean", parse_date("date_of_birth"))
    .withColumn("admission_date_clean", parse_date("admission_date"))
    .withColumn("discharge_date_clean", parse_date("discharge_date"))
)

**Gender**

In [0]:
df = df.withColumn(
    "gender_clean",
    F.when(F.upper(F.trim(F.col("gender"))).isin("M", "MALE",   "1"), F.lit("Male"))
     .when(F.upper(F.trim(F.col("gender"))).isin("F", "FEMALE", "0"), F.lit("Female"))
     .otherwise(F.lit(None).cast(StringType()))
)

**Numeric Values**

A sentinel value is a deliberate placeholder used instead of an actual NULL.

In [0]:
SENTINEL_NULLS = ["N/A", "NA", "NULL", "None", "", "-999"]

def clean_numeric(col_name, min_val=None, max_val=None):
    col = F.when(
        F.upper(F.trim(F.col(col_name))).isin(SENTINEL_NULLS), F.lit(None)
    ).otherwise(
        F.regexp_replace(F.col(col_name), r"[^0-9.\-]", "")
    ).cast(DoubleType())
    if min_val is not None:
        col = F.when(col < min_val, F.lit(None)).otherwise(col)
    if max_val is not None:
        col = F.when(col > max_val, F.lit(None)).otherwise(col)
    return col

df = (
    df
    .withColumn("height_cm_clean",          clean_numeric("height_cm",            100,  250))
    .withColumn("weight_kg_clean",           clean_numeric("weight_kg",             20,  300))
    .withColumn("heart_rate_bpm_clean",      clean_numeric("heart_rate_bpm",        20,  250))
    .withColumn("temperature_celsius_clean", clean_numeric("temperature_celsius",   34.0, 43.0))
    .withColumn("oxygen_saturation_clean",   clean_numeric("oxygen_saturation_pct", 50,  100))
    .withColumn("blood_glucose_clean",       clean_numeric("blood_glucose_mgdl",    20,  700))
    .withColumn("creatinine_clean",          clean_numeric("creatinine_mgdl",        0.1, 20.0))
    .withColumn("hba1c_clean",               clean_numeric("hba1c_pct",              3.0, 20.0))
)

**Age**

In [0]:
df = (
    df
    .withColumn("age_raw_clean", clean_numeric("age", 0, 130).cast(IntegerType()))
    .withColumn("age_computed",
        F.when(
            F.col("dob_clean").isNotNull() & F.col("admission_date_clean").isNotNull(),
            F.floor(F.months_between("admission_date_clean", "dob_clean") / 12)))
    .withColumn("age_clean", F.coalesce("age_computed", "age_raw_clean"))
)

**BMI**

In [0]:
df = (
    df
    .withColumn("bmi_raw_clean", clean_numeric("bmi", 10.0, 70.0))
    .withColumn("bmi_computed",
        F.when(
            F.col("height_cm_clean").isNotNull() &
            F.col("weight_kg_clean").isNotNull() &
            (F.col("height_cm_clean") > 0),
            F.round(
                F.col("weight_kg_clean") / F.pow(F.col("height_cm_clean") / 100, F.lit(2)), 2
            )
        ))
    .withColumn("bmi_clean", F.coalesce("bmi_computed", "bmi_raw_clean"))
)

**Length of stay**

In [0]:
df = (
    df
    .withColumn("los_computed",
        F.when(
            F.col("admission_date_clean").isNotNull() & F.col("discharge_date_clean").isNotNull(),
            F.datediff("discharge_date_clean", "admission_date_clean")
        ))
    .withColumn("los_clean",
        F.when(F.col("los_computed") >= 0, F.col("los_computed")))
    .withColumn("discharge_date_clean",
        F.when(F.col("los_computed") < 0, F.lit(None)).otherwise(F.col("discharge_date_clean")))
)


**Email**

In [0]:
df = (
    df
    .withColumn("email_lower", F.lower(F.trim(F.col("email"))))
    .withColumn("email_clean",
        F.when(
            F.col("email_lower").rlike(r"^[a-z0-9._%+\-]+@[a-z0-9.\-]+\.[a-z]{2,}$"),
            F.col("email_lower")
        ))
    .drop("email_lower")
)


**Insurance Provider**

In [0]:
df = df.withColumn(
    "insurance_provider_clean",
    F.when(
        F.upper(F.trim(F.col("insurance_provider"))).isin("UNKNOWN", "N/A", "NA", "", "NULL"),
        F.lit(None)
    ).otherwise(F.initcap(F.trim(F.col("insurance_provider"))))
)

**Claim Amount**

In [0]:
df = df.withColumn(
    "claim_amount_clean",
    F.when(F.col("insurance_claim_amount_usd").cast(DoubleType()) < 0, F.lit(None))
     .otherwise(F.col("insurance_claim_amount_usd").cast(DoubleType()))
)

**Smoker flag — normalise to Yes / No / Former**

In [0]:
df = df.withColumn(
    "smoker_clean",
    F.when(F.upper(F.trim(F.col("smoker"))).isin("YES", "Y", "1"), F.lit("Yes"))
     .when(F.upper(F.trim(F.col("smoker"))).isin("NO",  "N", "0"), F.lit("No"))
     .when(F.upper(F.trim(F.col("smoker"))) == "FORMER",           F.lit("Former"))
     .otherwise(F.lit(None))
)

In [0]:
display(df.take(10))


patient_id,first_name,last_name,date_of_birth,age,gender,ssn,phone_number,email,address,city,state,zip_code,admission_date,discharge_date,length_of_stay_days,admission_type,department,attending_physician,primary_diagnosis,secondary_diagnosis,medication_1,medication_2,height_cm,weight_kg,bmi,blood_pressure,heart_rate_bpm,temperature_celsius,oxygen_saturation_pct,blood_glucose_mgdl,creatinine_mgdl,hba1c_pct,insurance_provider,insurance_claim_amount_usd,icu_days,readmission_30day_flag,smoker,alcohol_use,discharge_disposition,_ingestion_timestamp,_source_file,_source_file_name,_file_modification_time,patient_id_clean,first_name_clean,last_name_clean,full_name,dob_clean,admission_date_clean,discharge_date_clean,gender_clean,height_cm_clean,weight_kg_clean,heart_rate_bpm_clean,temperature_celsius_clean,oxygen_saturation_clean,blood_glucose_clean,creatinine_clean,hba1c_clean,age_raw_clean,age_computed,age_clean,bmi_raw_clean,bmi_computed,bmi_clean,los_computed,los_clean,email_clean,insurance_provider_clean,claim_amount_clean,smoker_clean
10089,Christopher,Brown,1951-09-15,-999,M,261-65-8982,864-524-3854,CHRISTOPHER.BROWN91@AOL.COM,3658 Hill Way,Fort Worth,KY,90440,2021-10-27,2021-11-19,23,Transfer,Oncology,Dr. Hill,COPD,Alzheimer's Disease,Sertraline,null,N/A,57.0,17.3,157/83,97,38.6,97,null,3.02,N/A,CENTENE,33590.32,5,0,Y,null,Home,2026-05-06T00:57:24.816126Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z,10089,Christopher,Brown,Christopher Brown,1951-09-15,2021-10-27,2021-11-19,Male,null,57.0,97.0,38.6,97.0,null,3.02,null,null,70,70,17.3,null,17.3,23,23,christopher.brown91@aol.com,Centene,33590.32,Yes
10150,Donna,Carter,1977-01-27,null,F,114-80-7684,295-430-2858,donna.carter60@gmail.com,8166 Pine Ln,Philadelphia,OR,94896,2018-08-11,2018-08-18,7,Transfer,Oncology,Dr. Jones,Epilepsy,null,Aspirin,Albuterol,184.2,62.3,18.4,126/88,112,39.9,97,192,5.82,6.8,United Health,32913.02,3,1,no,Heavy,Rehab Facility,2026-05-06T00:57:24.816126Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z,10150,Donna,Carter,Donna Carter,1977-01-27,2018-08-11,2018-08-18,Female,184.2,62.3,112.0,39.9,97.0,192.0,5.82,6.8,null,41,41,18.4,18.36,18.36,7,7,donna.carter60@gmail.com,United Health,32913.02,No
10366,dorothy,null,1974-05-14,50,M,456-30-5262,977-411-9831,DOROTHY.ROBINSON63@YAHOO.COM,5328 Park Ln,Indianapolis,CA,96026,2024-09-15,2024-09-22,7,Urgent,Psychiatry,Dr. Carter,Atrial Fibrillation,Depression,Sertraline,null,191.6,null,23.8,139/109,100,38.7,95,null,1.96,4.7,null,50668.93,4,0,Unknown,Heavy,Transferred,2026-05-06T00:57:24.816126Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z,10366,Dorothy,null,Dorothy,1974-05-14,2024-09-15,2024-09-22,Male,191.6,null,100.0,38.7,95.0,null,1.96,4.7,50,50,50,23.8,null,23.8,7,7,dorothy.robinson63@yahoo.com,null,50668.93,null
10648,null,Campbell,1980-10-12,42,M,253-81-7912,213-276-2249,carol.campbell65@yahoo.com,1104 Main Ln,Baltimore,AZ,68271,2023-08-20,2023-08-29,9,Routine,Surgery,Dr. Adams,Asthma,Pneumonia,Amoxicillin,Albuterol,N/A,108.8,43.8,150/67,89,null,98,null,5.4,10.1,BlueCross BlueShield,10009.51,3,1,Unknown,Moderate,Home,2026-05-06T00:57:24.816126Z,dbfs:/Volumes/databricks_projects/patientrecord/my_volume/Patient_Record.parquet,Patient_Record.parquet,2026-05-06T00:51:41Z,10648,null,Campbell,Campbell,1980-10-12,2023-08-20,2023-08-29,Male,null,108.8,89.0,null,98.0,null,5.4,10.1,42,42,42,43.8,null,43.8,9,9,carol.campbell65@yahoo.com,Bluecross Blueshield,10009.51,null
10650,Kimberly,MARTINEZ,11/03/1988,32,F,685-70-4371,000-000-0000,kimberly.martinez42@yahoo.com,5745 Oak Ln,Philadelphia,OK,45938,2020-11-10,2020-12-09,29,Routine,ICU,Dr. Hall,Chronic Kidney Disease,Anemia,Amlodipine,null,187.9,N/A,24.9,172/90,111,38.6,100,314,3.64,6.3,Medicare,26408.67,4,1,N,None,AMA,2026-05-06T00:57:24.816126

**Silver Schema**

In [0]:
df_silver = df.select(
    F.col("patient_id_clean").alias("patient_id"),
    F.col("first_name_clean").alias("first_name"),
    F.col("last_name_clean").alias("last_name"),
    "full_name",
    F.col("dob_clean").alias("date_of_birth"),
    F.col("age_clean").alias("age"),
    F.col("gender_clean").alias("gender"),
    F.col("email_clean").alias("email"),
    "address", "city", "state",
    F.col("admission_date_clean").alias("admission_date"),
    F.col("discharge_date_clean").alias("discharge_date"),
    F.col("los_clean").alias("length_of_stay_days"),
    "admission_type", "department", "attending_physician",
    "primary_diagnosis",
    F.when(F.trim(F.col("secondary_diagnosis")) == "", F.lit(None))
     .otherwise(F.col("secondary_diagnosis")).alias("secondary_diagnosis"),
    "medication_1",
    F.when(F.trim(F.col("medication_2")) == "", F.lit(None))
     .otherwise(F.col("medication_2")).alias("medication_2"),
    F.col("height_cm_clean").alias("height_cm"),
    F.col("weight_kg_clean").alias("weight_kg"),
    F.col("bmi_clean").alias("bmi"),
    F.col("heart_rate_bpm_clean").alias("heart_rate_bpm"),
    F.col("temperature_celsius_clean").alias("temperature_celsius"),
    F.col("oxygen_saturation_clean").alias("oxygen_saturation_pct"),
    F.col("blood_glucose_clean").alias("blood_glucose_mgdl"),
    F.col("creatinine_clean").alias("creatinine_mgdl"),
    F.col("hba1c_clean").alias("hba1c_pct"),
    F.col("insurance_provider_clean").alias("insurance_provider"),
    F.col("claim_amount_clean").alias("insurance_claim_amount_usd"),
    F.col("icu_days").cast(IntegerType()),
    F.col("readmission_30day_flag").cast(IntegerType()),
    F.col("smoker_clean").alias("smoker"),
    "alcohol_use",
    F.current_timestamp().alias("_silver_timestamp"),
)

**Managed Table for silver layer**

In [0]:

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)           # managed table — Unity Catalog owns location
)

**Verification**

In [0]:
verify_df = spark.table(SILVER_TABLE)
print(f" Rows in {SILVER_TABLE}: {verify_df.count():,}")
display(verify_df.limit(5))


 Rows in databricks_projects.silver.patient_records_clean: 952


patient_id,first_name,last_name,full_name,date_of_birth,age,gender,email,address,city,state,admission_date,discharge_date,length_of_stay_days,admission_type,department,attending_physician,primary_diagnosis,secondary_diagnosis,medication_1,medication_2,height_cm,weight_kg,bmi,heart_rate_bpm,temperature_celsius,oxygen_saturation_pct,blood_glucose_mgdl,creatinine_mgdl,hba1c_pct,insurance_provider,insurance_claim_amount_usd,icu_days,readmission_30day_flag,smoker,alcohol_use,_silver_timestamp
10089,Christopher,Brown,Christopher Brown,1951-09-15,70,Male,christopher.brown91@aol.com,3658 Hill Way,Fort Worth,KY,2021-10-27,2021-11-19,23,Transfer,Oncology,Dr. Hill,COPD,Alzheimer's Disease,Sertraline,null,null,57.0,17.3,97.0,38.6,97.0,null,3.02,null,Centene,33590.32,5,0,Yes,null,2026-05-06T00:57:29.675263Z
10150,Donna,Carter,Donna Carter,1977-01-27,41,Female,donna.carter60@gmail.com,8166 Pine Ln,Philadelphia,OR,2018-08-11,2018-08-18,7,Transfer,Oncology,Dr. Jones,Epilepsy,null,Aspirin,Albuterol,184.2,62.3,18.36,112.0,39.9,97.0,192.0,5.82,6.8,United Health,32913.02,3,1,No,Heavy,2026-05-06T00:57:29.675263Z
10366,Dorothy,null,Dorothy,1974-05-14,50,Male,dorothy.robinson63@yahoo.com,5328 Park Ln,Indianapolis,CA,2024-09-15,2024-09-22,7,Urgent,Psychiatry,Dr. Carter,Atrial Fibrillation,Depression,Sertraline,null,191.6,null,23.8,100.0,38.7,95.0,null,1.96,4.7,null,50668.93,4,0,null,Heavy,2026-05-06T00:57:29.675263Z
10648,null,Campbell,Campbell,1980-10-12,42,Male,carol.campbell65@yahoo.com,1104 Main Ln,Baltimore,AZ,2023-08-20,2023-08-29,9,Routine,Surgery,Dr. Adams,Asthma,Pneumonia,Amoxicillin,Albuterol,null,108.8,43.8,89.0,null,98.0,null,5.4,10.1,Bluecross Blueshield,10009.51,3,1,null,Moderate,2026-05-06T00:57:29.675263Z
10650,Kimberly,Martinez,Kimberly Martinez,1988-11-03,32,Female,kimberly.martinez42@yahoo.com,5745 Oak Ln,Philadelphia,OK,2020-11-10,2020-12-09,29,Routine,ICU,Dr. Hall,Chronic Kidney Disease,Anemia,Amlodipine,null,187.9,null,24.9,111.0,38.6,100.0,314.0,3.64,6.3,Medicare,26408.67,4,1,No,None,2026-05-06T00:57:29.675263Z
